# Successful vs Unsuccessful Grant Comparison

Run the scoring pipeline on all PDFs in `data/successful/` and `data/unsuccessful/`, print results, and export to Excel.

In [ ]:
from __future__ import annotations

import json
import os
import sys
from pathlib import Path

import pandas as pd

PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

os.environ.setdefault('OLLAMA_HOST', 'http://localhost:11435')
os.environ.setdefault('OLLAMA_NUM_CTX', '131072')

from qwen3_ollama import _Scorer, score_application
from src.all_type_parser.all_type_parser import parse_and_save
from src.pool.build_pool import build_chunk_pool
from src.scoring.pipeline import (
    OVERALL_EXCLUDED_SECTIONS_BY_DOC_TYPE,
    SECTION_EXCLUDED_SUB_IDS_BY_DOC_TYPE,
    _aggregate_overall,
    _aggregate_section,
    _build_scored_section,
    _generate_json_with_parse_retry,
    _normalize_model_section_output,
    load_rubric,
)

CRITERIA_PATH = PROJECT_ROOT / 'criteria_points.json'
SUCCESSFUL_DIR = PROJECT_ROOT / 'data' / 'successful'
UNSUCCESSFUL_DIR = PROJECT_ROOT / 'data' / 'unsuccessful'
RESULTS_DIR = PROJECT_ROOT / 'experiments' / 'results' / 'compare'
PARSED_CACHE_DIR = RESULTS_DIR / 'parsed_cache'

EXPERIMENT_OLLAMA_MODEL = 'qwen3.5:27b'
BASELINE_MAX_TOKENS = int(os.environ.get('WHOLE_JSON_BASELINE_MAX_TOKENS', '65536'))

# Minimum character count of the full parsed JSON — PDFs below this are skipped
MIN_PARSED_CHARS = 2000

SECTION_KEYS = [
    'general',
    'proposed_research',
    'training_development',
    'sites_support',
    'wpcc',
    'application_form',
]

for path in [RESULTS_DIR, PARSED_CACHE_DIR]:
    path.mkdir(parents=True, exist_ok=True)

print('Project root      :', PROJECT_ROOT)
print('Results dir       :', RESULTS_DIR)
print('Ollama host       :', os.environ.get('OLLAMA_HOST'))
print('Ollama num_ctx    :', os.environ.get('OLLAMA_NUM_CTX'))
print('Model             :', EXPERIMENT_OLLAMA_MODEL)
print('Baseline max tok  :', BASELINE_MAX_TOKENS)
print('Min parsed chars  :', MIN_PARSED_CHARS)

In [ ]:
def parse_pdf_cached(pdf_path: Path, *, reparse: bool = False) -> tuple[dict, Path]:
    json_path = PARSED_CACHE_DIR / f'{pdf_path.stem}.json'
    if reparse or not json_path.exists():
        parse_and_save(str(pdf_path), str(json_path))
    raw = json_path.read_text(encoding='utf-8').strip()
    if not raw:
        raise ValueError(f'Parsed JSON is empty for {pdf_path.name}')
    if len(raw) < MIN_PARSED_CHARS:
        raise ValueError(
            f'Parsed JSON too short for {pdf_path.name}: {len(raw)} chars < {MIN_PARSED_CHARS} threshold'
        )
    parsed = json.loads(raw)
    return parsed, json_path


def score_pdf(pdf_path: Path, *, scorer: _Scorer, reparse: bool = False) -> dict:
    parsed, parsed_json_path = parse_pdf_cached(pdf_path, reparse=reparse)
    artifact_dir = RESULTS_DIR / 'artifacts' / pdf_path.stem
    artifact_dir.mkdir(parents=True, exist_ok=True)
    result = score_application(
        parsed, CRITERIA_PATH,
        doc_id=pdf_path.stem, scorer=scorer, artifacts_dir=artifact_dir,
    )
    return result


def _compact_text(text: str, limit: int = 900) -> str:
    compact = ' '.join((text or '').split())
    return compact[:limit] + '...' if len(compact) > limit else compact


_BASELINE_SYSTEM_PROMPT = """\
You are a strict NIHR grant reviewer performing one-shot scoring. \
You do NOT use staged retrieval, belief state, or multi-pass reasoning. \
Score the entire application in a single pass using the criteria, evidence_chunk_index, and parsed_application_json provided.

SCORING RULES
=============
1. Score EVERY signal for EVERY sub-criterion. Missing a signal is not allowed.
2. Each signal score is an integer 0–5.
3. used_chunk_ids must come from the evidence_chunk_index keys. \
Use up to 5 chunk_ids that most directly support your score for this sub-criterion. \
Use [] if no chunk is relevant.

OUTPUT FORMAT
=============
Return a single JSON object. Every section key and every sub-criterion key must be present. \
Example (two sub-criteria shown; your output must cover ALL criteria):

{
  "general": {
    "g.1": {
      "signals": {
        "g.1.a": 4,
        "g.1.b": 3,
        "g.1.c": 2,
        "g.1.d": 4
      },
      "used_chunk_ids": ["chunk_003", "chunk_017"]
    },
    "g.2": {
      "signals": {
        "g.2.a": 5,
        "g.2.b": 4,
        "g.2.c": 3
      },
      "used_chunk_ids": ["chunk_001"]
    }
  }
}
"""


def score_pdf_baseline(pdf_path: Path, *, scorer: _Scorer) -> dict:
    """One-shot baseline: full parsed JSON + chunk index, per-signal 0-5 scoring, pipeline aggregation."""
    parsed, _ = parse_pdf_cached(pdf_path)
    doc_type = (parsed.get('doc_type') or '').lower()
    excluded_sections = OVERALL_EXCLUDED_SECTIONS_BY_DOC_TYPE.get(doc_type, set())
    excluded_sub_ids = SECTION_EXCLUDED_SUB_IDS_BY_DOC_TYPE.get(doc_type, set())

    rubric_sections = load_rubric(CRITERIA_PATH)
    pool_data = build_chunk_pool(parsed)
    pool_lookup = pool_data['pool_lookup']
    all_chunk_ids = list(pool_lookup)
    chunk_order = {cid: idx for idx, cid in enumerate(all_chunk_ids)}

    chunk_index = [
        {'chunk_id': cid, 'parser_section': m.get('parser_section', ''),
         'preview': _compact_text(m.get('text', ''))}
        for cid, m in pool_lookup.items()
    ]

    chunk_id_item = {'type': 'string', 'enum': all_chunk_ids} if all_chunk_ids else {'type': 'string'}
    sec_props = {}
    for sec in rubric_sections:
        sub_props = {}
        for sub in sec['sub_criteria']:
            signal_props = {sig['sid']: {'type': 'integer', 'enum': [0, 1, 2, 3, 4, 5]} for sig in sub['signals']}
            sub_props[sub['sub_id']] = {
                'type': 'object',
                'properties': {
                    'signals': {'type': 'object', 'properties': signal_props,
                                'required': list(signal_props), 'additionalProperties': False},
                    'used_chunk_ids': {'type': 'array', 'items': chunk_id_item, 'maxItems': 5},
                },
                'required': ['signals', 'used_chunk_ids'],
                'additionalProperties': False,
            }
        sec_props[sec['section_key']] = {
            'type': 'object', 'properties': sub_props,
            'required': list(sub_props), 'additionalProperties': False,
        }
    schema = {'type': 'object', 'properties': sec_props,
              'required': list(sec_props), 'additionalProperties': False}

    messages = [
        {'role': 'system', 'content': _BASELINE_SYSTEM_PROMPT},
        {'role': 'user', 'content': json.dumps({
            'criteria': rubric_sections,
            'evidence_chunk_index': chunk_index,
            'parsed_application_json': parsed,
        }, ensure_ascii=False)},
    ]

    _, parsed_response, _ = _generate_json_with_parse_retry(
        scorer, messages, schema=schema, max_tokens=BASELINE_MAX_TOKENS, max_retries=0,
    )

    sections = []
    for rubric_section in rubric_sections:
        sk = rubric_section['section_key']
        raw_section = (parsed_response or {}).get(sk, {})
        if not isinstance(raw_section, dict):
            raw_section = {}
        normalized = _normalize_model_section_output(raw_section, rubric_section, all_chunk_ids)
        sections.append(_build_scored_section(rubric_section, normalized, chunk_order, excluded_sub_ids=excluded_sub_ids))

    features = {sec['section_key']: _aggregate_section(sec, pool_lookup) for sec in sections}
    section_weights = {sec['section_key']: sec['weight'] for sec in sections}
    overall = _aggregate_overall(features, section_weights, excluded_sections=excluded_sections)
    return {'overall': overall, 'features': features}


def get_overall(result: dict) -> float:
    return round(float(result.get('overall', {}).get('final_score_0to100', 0)), 2)


print('Helper functions defined.')

In [ ]:
# ── Run PIPELINE: SUCCESSFUL ──────────────────────────────────────────────────
PIPELINE_CSV_S = RESULTS_DIR / 'progress_pipeline_successful.csv'

scorer = _Scorer(model_name=EXPERIMENT_OLLAMA_MODEL)
successful_pdfs = sorted(SUCCESSFUL_DIR.glob('*.pdf'))
done_ps = set(pd.read_csv(PIPELINE_CSV_S)['pdf_name'].tolist()) if PIPELINE_CSV_S.exists() else set()
print(f'Pipeline successful: {len(done_ps)}/{len(successful_pdfs)} already done')

for idx, pdf_path in enumerate(successful_pdfs, start=1):
    if pdf_path.name in done_ps:
        print(f'[{idx}/{len(successful_pdfs)}] SKIP {pdf_path.name}'); continue
    print(f'\n[{idx}/{len(successful_pdfs)}] {pdf_path.name}')
    try:
        score = get_overall(score_pdf(pdf_path, scorer=scorer))
        row = {'pdf_name': pdf_path.name, 'category': 'successful', 'pipeline_overall': score}
        pd.DataFrame([row]).to_csv(PIPELINE_CSV_S, mode='a', header=not PIPELINE_CSV_S.exists(), index=False)
        done_ps.add(pdf_path.name)
        print(f'  pipeline={score:.1f}')
    except Exception as exc:
        print(f'  ERROR: {exc}')

print(f'\nDone. {len(done_ps)}/{len(successful_pdfs)} successful (pipeline)')

In [ ]:
# ── Run BASELINE: SUCCESSFUL ──────────────────────────────────────────────────
BASELINE_CSV_S = RESULTS_DIR / 'progress_baseline_successful.csv'

scorer = _Scorer(model_name=EXPERIMENT_OLLAMA_MODEL)
successful_pdfs = sorted(SUCCESSFUL_DIR.glob('*.pdf'))
done_bs = set(pd.read_csv(BASELINE_CSV_S)['pdf_name'].tolist()) if BASELINE_CSV_S.exists() else set()
print(f'Baseline successful: {len(done_bs)}/{len(successful_pdfs)} already done')

for idx, pdf_path in enumerate(successful_pdfs, start=1):
    if pdf_path.name in done_bs:
        print(f'[{idx}/{len(successful_pdfs)}] SKIP {pdf_path.name}'); continue
    print(f'\n[{idx}/{len(successful_pdfs)}] {pdf_path.name}')
    try:
        score = get_overall(score_pdf_baseline(pdf_path, scorer=scorer))
        row = {'pdf_name': pdf_path.name, 'category': 'successful', 'baseline_overall': score}
        pd.DataFrame([row]).to_csv(BASELINE_CSV_S, mode='a', header=not BASELINE_CSV_S.exists(), index=False)
        done_bs.add(pdf_path.name)
        print(f'  baseline={score:.1f}')
    except Exception as exc:
        print(f'  ERROR: {exc}')

print(f'\nDone. {len(done_bs)}/{len(successful_pdfs)} successful (baseline)')

In [ ]:
# ── Run PIPELINE: UNSUCCESSFUL ────────────────────────────────────────────────
PIPELINE_CSV_U = RESULTS_DIR / 'progress_pipeline_unsuccessful.csv'

scorer = _Scorer(model_name=EXPERIMENT_OLLAMA_MODEL)
unsuccessful_pdfs = sorted(UNSUCCESSFUL_DIR.glob('*.pdf'))
done_pu = set(pd.read_csv(PIPELINE_CSV_U)['pdf_name'].tolist()) if PIPELINE_CSV_U.exists() else set()
print(f'Pipeline unsuccessful: {len(done_pu)}/{len(unsuccessful_pdfs)} already done')

for idx, pdf_path in enumerate(unsuccessful_pdfs, start=1):
    if pdf_path.name in done_pu:
        print(f'[{idx}/{len(unsuccessful_pdfs)}] SKIP {pdf_path.name}'); continue
    print(f'\n[{idx}/{len(unsuccessful_pdfs)}] {pdf_path.name}')
    try:
        score = get_overall(score_pdf(pdf_path, scorer=scorer))
        row = {'pdf_name': pdf_path.name, 'category': 'unsuccessful', 'pipeline_overall': score}
        pd.DataFrame([row]).to_csv(PIPELINE_CSV_U, mode='a', header=not PIPELINE_CSV_U.exists(), index=False)
        done_pu.add(pdf_path.name)
        print(f'  pipeline={score:.1f}')
    except Exception as exc:
        print(f'  ERROR: {exc}')

print(f'\nDone. {len(done_pu)}/{len(unsuccessful_pdfs)} unsuccessful (pipeline)')

In [ ]:
# ── Run BASELINE: UNSUCCESSFUL ────────────────────────────────────────────────
BASELINE_CSV_U = RESULTS_DIR / 'progress_baseline_unsuccessful.csv'

scorer = _Scorer(model_name=EXPERIMENT_OLLAMA_MODEL)
unsuccessful_pdfs = sorted(UNSUCCESSFUL_DIR.glob('*.pdf'))
done_bu = set(pd.read_csv(BASELINE_CSV_U)['pdf_name'].tolist()) if BASELINE_CSV_U.exists() else set()
print(f'Baseline unsuccessful: {len(done_bu)}/{len(unsuccessful_pdfs)} already done')

for idx, pdf_path in enumerate(unsuccessful_pdfs, start=1):
    if pdf_path.name in done_bu:
        print(f'[{idx}/{len(unsuccessful_pdfs)}] SKIP {pdf_path.name}'); continue
    print(f'\n[{idx}/{len(unsuccessful_pdfs)}] {pdf_path.name}')
    try:
        score = get_overall(score_pdf_baseline(pdf_path, scorer=scorer))
        row = {'pdf_name': pdf_path.name, 'category': 'unsuccessful', 'baseline_overall': score}
        pd.DataFrame([row]).to_csv(BASELINE_CSV_U, mode='a', header=not BASELINE_CSV_U.exists(), index=False)
        done_bu.add(pdf_path.name)
        print(f'  baseline={score:.1f}')
    except Exception as exc:
        print(f'  ERROR: {exc}')

print(f'\nDone. {len(done_bu)}/{len(unsuccessful_pdfs)} unsuccessful (baseline)')

In [ ]:
# ── Analysis & Export ─────────────────────────────────────────────────────────
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import matplotlib.patches as mpatches
import numpy as np

# Merge all four CSVs
df_ps = pd.read_csv(PIPELINE_CSV_S)
df_bs = pd.read_csv(BASELINE_CSV_S).drop(columns=['category'], errors='ignore')
df_pu = pd.read_csv(PIPELINE_CSV_U)
df_bu = pd.read_csv(BASELINE_CSV_U).drop(columns=['category'], errors='ignore')

df_s = df_ps.merge(df_bs, on='pdf_name', how='outer')
df_u = df_pu.merge(df_bu, on='pdf_name', how='outer')
df = pd.concat([df_s, df_u], ignore_index=True).sort_values('pdf_name').reset_index(drop=True)

display(df[['pdf_name', 'category', 'pipeline_overall', 'baseline_overall']])

# ── Summary table ─────────────────────────────────────────────────────────────
summary = df.groupby('category')[['pipeline_overall', 'baseline_overall']].agg(['mean', 'std']).round(2)
print('\nGroup summary:')
display(summary)

# ── Box plot ──────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(10, 5), sharey=True)
fig.suptitle('Pipeline vs Baseline — Successful vs Unsuccessful', fontsize=13)

groups = ['successful', 'unsuccessful']
colors = ['#4C72B0', '#DD8452']

for ax_idx, (ax, method, col) in enumerate(zip(axes, ['Pipeline', 'Baseline'], ['pipeline_overall', 'baseline_overall'])):
    data = [df.loc[df['category'] == g, col].dropna().values for g in groups]
    bp = ax.boxplot(data, patch_artist=True, widths=0.5,
                    medianprops=dict(color='black', linewidth=2))
    for patch, color in zip(bp['boxes'], colors):
        patch.set_facecolor(color)
        patch.set_alpha(0.7)

    for i, (d, color) in enumerate(zip(data, colors), start=1):
        ax.scatter(np.full(len(d), i) + np.random.uniform(-0.08, 0.08, len(d)),
                   d, color=color, s=30, zorder=3, alpha=0.8)

    means = [d.mean() if len(d) else float('nan') for d in data]
    ax.scatter([1, 2], means, marker='D', color='white', edgecolors='black', s=50, zorder=4)

    ax.set_title(method)
    ax.set_xticks([1, 2])
    ax.set_xticklabels(groups)
    ax.set_ylabel('Score (0–100)')
    ax.yaxis.set_minor_locator(mticker.AutoMinorLocator())
    ax.grid(axis='y', alpha=0.3)

    for i, m in enumerate(means, start=1):
        if not np.isnan(m):
            ax.text(i, m + 1.5, f'{m:.1f}', ha='center', va='bottom', fontsize=9, fontweight='bold')

    if ax_idx == 0:
        handles = [mpatches.Patch(facecolor=c, alpha=0.7, label=g) for c, g in zip(colors, groups)]
        ax.legend(handles=handles, loc='upper left', fontsize=9)

plt.tight_layout()
ts = pd.Timestamp.now().strftime('%Y%m%d_%H%M%S')
fig_path = RESULTS_DIR / f'compare_{ts}.png'
plt.savefig(fig_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved: {fig_path}')

# ── Excel export ───────────────────────────────────────────────────────────────
excel_path = RESULTS_DIR / f'compare_{ts}.xlsx'
with pd.ExcelWriter(excel_path) as writer:
    df[['pdf_name', 'category', 'pipeline_overall', 'baseline_overall']].to_excel(
        writer, sheet_name='Scores', index=False)
    summary.to_excel(writer, sheet_name='Summary')
print(f'Exported: {excel_path}')

In [ ]:
# ── Distribution Comparison: Pipeline vs Baseline ─────────────────────────────
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

# ── Load & merge all results ──────────────────────────────────────────────────
def load_merged(pipeline_csv: Path, baseline_csv: Path) -> pd.DataFrame:
    df_p = pd.read_csv(pipeline_csv)
    df_b = pd.read_csv(baseline_csv).drop(columns=['category'], errors='ignore')
    return df_p.merge(df_b, on='pdf_name', how='inner')

df_s = load_merged(PIPELINE_CSV_S, BASELINE_CSV_S).assign(group='successful')
df_u = load_merged(PIPELINE_CSV_U, BASELINE_CSV_U).assign(group='unsuccessful')
df_all = pd.concat([df_s, df_u], ignore_index=True)

successful_mask = df_all['group'] == 'successful'
unsuccessful_mask = df_all['group'] == 'unsuccessful'

COLORS = {'successful': '#2ecc71', 'unsuccessful': '#e74c3c'}
ALPHA = 0.55
BINS = 10

plot_cols = ['overall'] + SECTION_KEYS[:3]
col_labels = ['Overall', 'General', 'Proposed Research', 'Training & Dev']

fig, axes = plt.subplots(2, 4, figsize=(18, 7), sharey=False)
fig.suptitle('Score Distributions: Successful vs Unsuccessful\n'
             'Top = Pipeline (our method)  |  Bottom = Baseline (one-shot)',
             fontsize=13, fontweight='bold', y=1.01)

for row, method in enumerate(['pipeline', 'baseline']):
    for col, (key, label) in enumerate(zip(plot_cols, col_labels)):
        ax = axes[row, col]
        col_name = f'{method}_overall_score_100' if key == 'overall' else f'{method}_{key}_score_100'

        s_scores = df_all.loc[successful_mask, col_name].dropna()
        u_scores = df_all.loc[unsuccessful_mask, col_name].dropna()

        bin_edges = np.linspace(0, 100, BINS + 1)

        ax.hist(s_scores, bins=bin_edges, alpha=ALPHA, color=COLORS['successful'], label='Successful')
        ax.hist(u_scores, bins=bin_edges, alpha=ALPHA, color=COLORS['unsuccessful'], label='Unsuccessful')

        ax.axvline(s_scores.mean(), color=COLORS['successful'], lw=2, ls='--')
        ax.axvline(u_scores.mean(), color=COLORS['unsuccessful'], lw=2, ls='--')

        diff = s_scores.mean() - u_scores.mean()
        ax.set_title(f'{label}\nΔmean = {diff:+.1f}', fontsize=10)
        ax.set_xlabel('Score (0–100)', fontsize=8)
        ax.set_xlim(0, 100)
        if col == 0:
            ax.set_ylabel('Pipeline' if row == 0 else 'Baseline', fontsize=10, fontweight='bold')

        ax.legend(loc='upper left', fontsize=7.5)

plt.tight_layout()
plt.savefig(RESULTS_DIR / 'distribution_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: distribution_comparison.png')

# ── Summary: mean scores per section, side-by-side bars ──────────────────────
fig2, axes2 = plt.subplots(1, 2, figsize=(16, 5), sharey=True)
fig2.suptitle('Mean Section Scores by Group', fontsize=13, fontweight='bold')

x = np.arange(len(SECTION_KEYS) + 1)
tick_labels = ['Overall'] + [k.replace('_', '\n') for k in SECTION_KEYS]
width = 0.35

for ax, method in zip(axes2, ['pipeline', 'baseline']):
    cols = [f'{method}_overall_score_100'] + [f'{method}_{k}_score_100' for k in SECTION_KEYS]
    s_means = df_all.loc[successful_mask, cols].mean().values
    u_means = df_all.loc[unsuccessful_mask, cols].mean().values

    ax.bar(x - width/2, s_means, width, color=COLORS['successful'], alpha=0.8, label='Successful')
    ax.bar(x + width/2, u_means, width, color=COLORS['unsuccessful'], alpha=0.8, label='Unsuccessful')

    ax.set_title(f'{"Pipeline (ours)" if method == "pipeline" else "Baseline (one-shot)"}',
                 fontsize=11, fontweight='bold')
    ax.set_xticks(x)
    ax.set_xticklabels(tick_labels, fontsize=9)
    ax.set_ylabel('Mean Score (0–100)')
    ax.set_ylim(0, 100)
    ax.grid(axis='y', alpha=0.3)
    ax.legend(loc='upper left', fontsize=9)

    for i, (sm, um) in enumerate(zip(s_means, u_means)):
        ax.text(x[i], max(sm, um) + 2, f'Δ{sm - um:+.1f}',
                ha='center', fontsize=7.5, color='#333333')

plt.tight_layout()
plt.savefig(RESULTS_DIR / 'mean_scores_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: mean_scores_comparison.png')

# ── Print summary table ───────────────────────────────────────────────────────
print('\n── Mean scores (successful vs unsuccessful) ──')
rows = []
for method in ['pipeline', 'baseline']:
    for key in ['overall'] + SECTION_KEYS:
        col = f'{method}_overall_score_100' if key == 'overall' else f'{method}_{key}_score_100'
        s_m = df_all.loc[successful_mask, col].mean()
        u_m = df_all.loc[unsuccessful_mask, col].mean()
        rows.append({'method': method, 'section': key,
                     'successful_mean': round(s_m, 1), 'unsuccessful_mean': round(u_m, 1),
                     'delta': round(s_m - u_m, 1)})
summary_df = pd.DataFrame(rows)
display(summary_df.pivot(index='section', columns='method', values=['successful_mean', 'unsuccessful_mean', 'delta']))